## ETL of GeoSpatial data from US Census to Hugging Face

The US Census has restrictions on how it serves files.  So we'll copy them elsewhere first.


## Setup


In [1]:
%%capture
%%bash
apt-get update
apt-get install -y tree jq


In [2]:
import os
import duckdb
import pandas as pd
from google.colab import userdata


## Download all shapefiles


In [3]:
%%bash

hf download rwcitek/DUC-2026 \
  --repo-type dataset \
  --include "TIGER2024/**" \
  --local-dir ./


✓ Downloaded
  path: /content


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Fetching 3234 files:   0%|          | 0/3234 [00:00<?, ?it/s]Still waiting to acquire lock on /content/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /content/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /content/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 3234 files: 100%|██████████| 3234/3234 [02:48<00:00, 19.16it/s]


In [4]:
! find TIGER2024/ | grep -c zip$


3234


## Convert shapefiles to parquet




### State



In [5]:
zip_path = "TIGER2024/STATE/tl_2024_us_state.zip"
out_path = "parquet/TIGER2024/STATE"
out_parquet = f"{out_path}/tl_2024_us_state.parquet"


In [6]:
! mkdir -p {out_path}


In [7]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")


In [8]:
con.execute(
    f"""
    COPY (
        SELECT *
        FROM ST_Read('/vsizip/{zip_path}')
    )
    TO '{out_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD);
    """
)

In [9]:
! find {out_path} -ls


     4569      4 drwxr-xr-x   2 root     root         4096 Nov 22 00:14 parquet/TIGER2024/STATE
   262167  11076 -rw-r--r--   1 root     root     11340375 Nov 22 00:14 parquet/TIGER2024/STATE/tl_2024_us_state.parquet


### Roads


### Clean up zip files


In [11]:
bad_files = ! find ./TIGER2024/ | grep zip$ | xargs -n 1 -P 40 file | grep -v Zip
bad_files


['./TIGER2024/ROADS/tl_2024_48211_roads.zip: HTML document, ASCII text, with no line terminators',
 './TIGER2024/ROADS/tl_2024_06079_roads.zip: HTML document, ASCII text, with no line terminators']

***Note that not every file that ends in ".zip" is a zip archive file.***


In [20]:
# Remove bad zip files
for i in bad_files:
  name = i.split(":")[0]
  print(name, i)
  ! ls -la {name}
  ! rm -f {name}
  ! ls -la {name}


./TIGER2024/ROADS/tl_2024_48211_roads.zip ./TIGER2024/ROADS/tl_2024_48211_roads.zip: HTML document, ASCII text, with no line terminators
-rw-r--r-- 1 root root 247 Nov 22 00:14 ./TIGER2024/ROADS/tl_2024_48211_roads.zip
ls: cannot access './TIGER2024/ROADS/tl_2024_48211_roads.zip': No such file or directory
./TIGER2024/ROADS/tl_2024_06079_roads.zip ./TIGER2024/ROADS/tl_2024_06079_roads.zip: HTML document, ASCII text, with no line terminators
-rw-r--r-- 1 root root 247 Nov 22 00:12 ./TIGER2024/ROADS/tl_2024_06079_roads.zip
ls: cannot access './TIGER2024/ROADS/tl_2024_06079_roads.zip': No such file or directory


In [21]:
# Verify that the bad files are gone
bad_files = ! find ./TIGER2024/ | grep zip$ | xargs -n 1 -P 40 file | grep -v Zip
bad_files


[]

#### Get a list of all shape files


In [22]:
roads_shapefiles = ! find ./TIGER2024/ROADS -name '*.zip' | sort
len(roads_shapefiles)


3231

In [23]:
roads_columns = []
for i, road in enumerate(roads_shapefiles[:]):
  if i % 100 == 0 : print(i, road)
  df = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM ST_Read('/vsizip/{road}')
  """).df()
  roads_columns.append([road, df["column_name"].values.tolist() ])

len(roads_columns)


0 ./TIGER2024/ROADS/tl_2024_01001_roads.zip
100 ./TIGER2024/ROADS/tl_2024_04007_roads.zip
200 ./TIGER2024/ROADS/tl_2024_06027_roads.zip
300 ./TIGER2024/ROADS/tl_2024_08111_roads.zip
400 ./TIGER2024/ROADS/tl_2024_13025_roads.zip
500 ./TIGER2024/ROADS/tl_2024_13229_roads.zip
600 ./TIGER2024/ROADS/tl_2024_17009_roads.zip
700 ./TIGER2024/ROADS/tl_2024_18005_roads.zip
800 ./TIGER2024/ROADS/tl_2024_19021_roads.zip
900 ./TIGER2024/ROADS/tl_2024_20023_roads.zip
1000 ./TIGER2024/ROADS/tl_2024_21013_roads.zip
1100 ./TIGER2024/ROADS/tl_2024_21213_roads.zip
1200 ./TIGER2024/ROADS/tl_2024_24015_roads.zip
1300 ./TIGER2024/ROADS/tl_2024_26137_roads.zip
1400 ./TIGER2024/ROADS/tl_2024_27171_roads.zip
1500 ./TIGER2024/ROADS/tl_2024_29033_roads.zip
1600 ./TIGER2024/ROADS/tl_2024_30003_roads.zip
1700 ./TIGER2024/ROADS/tl_2024_31091_roads.zip
1800 ./TIGER2024/ROADS/tl_2024_35007_roads.zip
1900 ./TIGER2024/ROADS/tl_2024_37019_roads.zip
2000 ./TIGER2024/ROADS/tl_2024_38019_roads.zip
2100 ./TIGER2024/ROADS/tl

3231

In [28]:
roads_df = pd.DataFrame( roads_columns, columns = ["shapefile", "schema"] )
roads_df


,shapefile,schema
0,./TIGER2024/ROADS/tl_2024_01001_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
1,./TIGER2024/ROADS/tl_2024_01003_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
2,./TIGER2024/ROADS/tl_2024_01005_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
3,./TIGER2024/ROADS/tl_2024_01007_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
4,./TIGER2024/ROADS/tl_2024_01009_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
...,...,...
3226,./TIGER2024/ROADS/tl_2024_72151_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
3227,./TIGER2024/ROADS/tl_2024_72153_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
3228,./TIGER2024/ROADS/tl_2024_78010_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
3229,./TIGER2024/ROADS/tl_2024_78020_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"


### Verify that the schema is the same for all shapefiles


In [29]:
duckdb.query(f'''
  select schema, count(schema) as count
  from roads_df
  group by all
  ;
''')


┌──────────────────────────────────────────┬───────┐
│                  schema                  │ count │
│                varchar[]                 │ int64 │
├──────────────────────────────────────────┼───────┤
│ [LINEARID, FULLNAME, RTTYP, MTFCC, geom] │  3231 │
└──────────────────────────────────────────┴───────┘

In [35]:
duckdb.query(f'''
  INSTALL spatial;
  LOAD spatial;

  select typeof(geom)
  FROM ST_Read('/vsizip/./TIGER2024/ROADS/tl_2024_01001_roads.zip')
  limit 1
  ;
''')

┌──────────────┐
│ typeof(geom) │
│   varchar    │
├──────────────┤
│ GEOMETRY     │
└──────────────┘

In [ ]:
# Wait for roads downloads to finish
%%capture
%%bash

while [ -f /tmp/tiger.roads.sh ] ; do
  date
  sleep 1
done


### Verify


In [ ]:
! tree TIGER2024/


In [ ]:
! find ./TIGER2024/ | grep -c zip$


# Keep


## Push to Hugging Face


## Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


In [ ]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


In [ ]:
hf_file = os.environ["HF_FILE"] = "TIGER2024/"
hf_file


In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


## Push the parquet file to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


In [ ]:
%%capture hf_repo_list
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf datasets  \
  list -R \
  "${TARGET_REPO}"/ \


In [ ]:
print(hf_repo_list.stdout)
